# Ch.5 — Stacking & Blending

## EnsembleAI Challenge — Ch.5

Combine diverse model families via a **meta-learner** trained on out-of-fold predictions.

Key question: does stacking beat the best individual model?

## Setup & Data

In [ ]:
# ── Setup & Data ─────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import (RandomForestRegressor, StackingRegressor,
                               RandomForestClassifier, StackingClassifier)
from sklearn.linear_model import RidgeCV, LogisticRegressionCV
from sklearn.metrics import mean_squared_error, r2_score, f1_score
from xgboost import XGBRegressor, XGBClassifier

IMG = "img/"
import os; os.makedirs(IMG, exist_ok=True)

np.random.seed(42)

data = fetch_california_housing()
X, y = data.data, data.target
y_cls = (y > np.median(y)).astype(int)

X_tr, X_te, y_tr, y_te, y_tr_cls, y_te_cls = train_test_split(
    X, y, y_cls, test_size=0.2, random_state=42)

print(f"Train: {len(X_tr):,}  Test: {len(X_te):,}")

## Individual Model Baselines

In [ ]:
# ── Individual Model Baselines ───────────────────────────────────────────────
models = {
    'Ridge': RidgeCV(),
    'RF (200)': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4,
                             random_state=42, verbosity=0),
}

individual_rmses = {}
for name, model in models.items():
    t0 = time.time()
    model.fit(X_tr, y_tr)
    t = time.time() - t0
    rmse = np.sqrt(mean_squared_error(y_te, model.predict(X_te)))
    individual_rmses[name] = rmse
    print(f"{name:>12}: RMSE={rmse:.4f}  Time={t:.2f}s")

## Stacking (Cross-Validated)

In [ ]:
# ── Stacking Regression ──────────────────────────────────────────────────────
stack_reg = StackingRegressor(
    estimators=[
        ('ridge', RidgeCV()),
        ('rf', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
        ('xgb', XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4,
                              random_state=42, verbosity=0)),
    ],
    final_estimator=RidgeCV(),
    cv=5, n_jobs=-1)

t0 = time.time()
stack_reg.fit(X_tr, y_tr)
t_stack = time.time() - t0

rmse_stack = np.sqrt(mean_squared_error(y_te, stack_reg.predict(X_te)))
best_individual = min(individual_rmses.values())
gain = (best_individual - rmse_stack) / best_individual * 100

print(f"Stacked Ensemble RMSE: {rmse_stack:.4f}  Time: {t_stack:.2f}s")
print(f"Best individual RMSE:  {best_individual:.4f}")
print(f"Stack improvement:     {gain:.1f}%")

## Comparison Chart

In [ ]:
# ── Comparison ───────────────────────────────────────────────────────────────
all_rmses = {**individual_rmses, 'Stacked': rmse_stack}

fig, ax = plt.subplots(figsize=(9, 4))
names = list(all_rmses.keys())
values = list(all_rmses.values())
colors = ['#aec6e8', '#6baed6', '#2171b5', '#08306b']
ax.bar(names, values, color=colors, edgecolor='white')
ax.set_ylabel('Test RMSE'); ax.set_title('Individual Models vs Stacked Ensemble', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(f'{IMG}ch05_stacking_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Meta-Learner Weights

In [ ]:
# ── Meta-Learner Coefficients ────────────────────────────────────────────────
meta = stack_reg.final_estimator_
base_names = ['Ridge', 'RF', 'XGBoost']

print("Meta-learner (Ridge) coefficients:")
for name, coef in zip(base_names, meta.coef_):
    print(f"  {name:>10}: {coef:.4f}")
print(f"  {'Intercept':>10}: {meta.intercept_:.4f}")
print("\nHigher coefficient = meta-learner trusts this model more.")

## Stacking — Classification

In [ ]:
# ── Stacking Classification ──────────────────────────────────────────────────
stack_cls = StackingClassifier(
    estimators=[
        ('lr', LogisticRegressionCV(max_iter=1000)),
        ('rf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
        ('xgb', XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=4,
                               random_state=42, verbosity=0, eval_metric='logloss')),
    ],
    final_estimator=LogisticRegressionCV(max_iter=1000),
    cv=5, n_jobs=-1)

stack_cls.fit(X_tr, y_tr_cls)
f1_stack = f1_score(y_te_cls, stack_cls.predict(X_te))

# Individual baselines
for name, model in [('RF', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
                     ('XGBoost', XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=4,
                                                random_state=42, verbosity=0, eval_metric='logloss'))]:
    model.fit(X_tr, y_tr_cls)
    print(f"{name:>10} F1: {f1_score(y_te_cls, model.predict(X_te)):.4f}")
print(f"{'Stacked':>10} F1: {f1_stack:.4f}")

## Simple Average vs Stacking

In [ ]:
# ── Simple Average (no meta-learner) ─────────────────────────────────────────
# Reuse individually trained models from baseline
avg_pred = np.mean([
    models['Ridge'].predict(X_te),
    models['RF (200)'].predict(X_te),
    models['XGBoost'].predict(X_te),
], axis=0)

rmse_avg = np.sqrt(mean_squared_error(y_te, avg_pred))
print(f"Simple average RMSE: {rmse_avg:.4f}")
print(f"Stacked RMSE:        {rmse_stack:.4f}")
print(f"Best individual:     {best_individual:.4f}")
print(f"\nSimple average is a strong baseline — stacking adds learned weighting.")

## Progress Check

| # | Constraint | Status | Evidence |
|---|-----------|--------|----------|
| 1 | IMPROVEMENT | ✅ | Stack beats best individual |
| 2 | DIVERSITY | ✅ | Ridge + RF + XGBoost = different families |
| 3 | EFFICIENCY | ⚠️ | 3× inference time |
| 4 | INTERPRETABILITY | ✅ | SHAP on meta-learner (Ch.4) |
| 5 | ROBUSTNESS | ✅ | Cross-validated OOF predictions |

**Next**: Ch.6 — Production ensembles: latency budgets, pruning, A/B testing.

## Exercises

1. **Diversity matters**: Stack two XGBoost models with different seeds. Compare improvement vs stacking XGBoost + RF + Ridge. Prove that diversity is the key.
2. **Meta-learner choice**: Replace RidgeCV with XGBRegressor(max_depth=2) as the meta-learner. Does it help or hurt?
3. **Blending (manual)**: Implement blending — split training into 70/30, train base models on 70%, predict on 30%, train meta-learner on those predictions. Compare to CV stacking.

In [ ]:
# ── Exercise 1: Diversity experiment ─────────────────────────────────────────
# TODO: Stack XGBoost(seed=42) + XGBoost(seed=123)
#     vs Stack XGBoost + RF + Ridge
#     Compare RMSE improvement
pass

In [ ]:
# ── Exercise 2: Meta-learner choice ──────────────────────────────────────────
# TODO: Replace RidgeCV() with XGBRegressor(max_depth=2) as final_estimator
#     Compare RMSE
pass

In [ ]:
# ── Exercise 3: Manual blending ──────────────────────────────────────────────
# TODO: X_base, X_blend = train_test_split(X_tr, test_size=0.3)
#     Train base models on X_base
#     Predict on X_blend → meta-features
#     Train Ridge on meta-features
#     Compare to StackingRegressor
pass